In [2]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib  # For loading models if saved

# --- Load Data ---
# Assuming the test data (X_test, y_test) is prepared in the same way as before.
# Load test set (if not already loaded in memory)
data = pd.read_csv('/Users/suryanshu/Downloads/SPBSS5IP_01122020_25122024.csv', parse_dates=['Date'])
data.set_index('Date', inplace=True)

# Feature Engineering (reuse from previous steps)
data['MA_10'] = data['Close'].rolling(window=10).mean()
data['MA_20'] = data['Close'].rolling(window=20).mean()
data['Daily_Return'] = data['Close'].pct_change()
data['Volatility'] = data['Close'].rolling(window=10).std()
data['Lag_1'] = data['Close'].shift(1)
data['Lag_2'] = data['Close'].shift(2)
data.dropna(inplace=True)

features = ['Open', 'High', 'Low', 'Close', 'MA_10', 'MA_20', 'Daily_Return', 'Volatility', 'Lag_1', 'Lag_2']
X = data[features]
y = data['Close']

# Scale Features
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train-Test Split
train_size = int(0.8 * len(X_scaled))
X_test = X_scaled[train_size:]
y_test = y.values[train_size:]

# --- Load Models (if needed) ---
# Random Forest
rf_model = joblib.load('rf_model.pkl')  # Uncomment if saved
# LightGBM
lgb_model = joblib.load('lgb_model.pkl')  # Uncomment if saved

# If models are already in memory, skip this step.

# --- Generate Predictions ---
rf_predictions = rf_model.predict(X_test)
lgb_predictions = lgb_model.predict(X_test)

# --- Evaluate Metrics ---
def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"Model: {name}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  R²: {r2:.2f}\n")
    return rmse, mae, r2

# Evaluate Random Forest
rf_metrics = evaluate_model("Random Forest", y_test, rf_predictions)

# Evaluate LightGBM
lgb_metrics = evaluate_model("LightGBM", y_test, lgb_predictions)

# --- Visualization ---
# Plot Actual vs Predicted Prices
plt.figure(figsize=(14, 7))
plt.plot(y_test, label='Actual Prices', color='blue', alpha=0.6)
plt.plot(rf_predictions, label='Random Forest Predictions', color='orange', alpha=0.8)
plt.plot(lgb_predictions, label='LightGBM Predictions', color='green', alpha=0.8)
plt.title('Actual vs Predicted Prices')
plt.xlabel('Time')
plt.ylabel('Stock Price')
plt.legend()
plt.show()

# Residual Plots
plt.figure(figsize=(14, 7))
plt.scatter(y_test, y_test - rf_predictions, color='orange', label='Random Forest Residuals', alpha=0.6)
plt.scatter(y_test, y_test - lgb_predictions, color='green', label='LightGBM Residuals', alpha=0.6)
plt.axhline(y=0, color='blue', linestyle='--', alpha=0.8)
plt.title('Residuals Plot')
plt.xlabel('Actual Prices')
plt.ylabel('Residuals')
plt.legend()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'rf_model.pkl'